# Quantizing LLMs to GGUF Format

This notebook demonstrates the process of quantizing a large language model (Qwen3.5-2B) to GGUF format. The GGUF format allows models to run efficiently on various hardware by reducing model size while maintaining reasonable performance.

## Overview of Steps:
1. Clone the llama.cpp repository
2. Install required dependencies
3. Download the model from Hugging Face
4. Configure paths and quantization methods
5. Convert the model to GGUF format
6. Build the llama.cpp project
7. Quantize the model
8. Test the quantized model
9. Upload to Hugging Face Hub

## Step 1: Clone llama.cpp Repository

We clone the llama.cpp repository which provides the tools needed to convert and quantize models to GGUF format.

In [1]:
!git clone https://github.com/ggml-org/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 88424, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 88424 (delta 42), reused 23 (delta 23), pack-reused 88344 (from 2)
Receiving objects: 100% (88424/88424), 372.33 MiB | 33.46 MiB/s, done.
Resolving deltas: 100% (62703/62703), done.


## Step 2: Install Required Dependencies

Install necessary Python packages including setuptools, wheel, numpy, PyYAML, requests, pydantic, and openai. These are required for model conversion and quantization tasks.

In [2]:
!pip install setuptools wheel && pip install --only-binary :all: numpy pyyaml requests pydantic openai

## Step 3: Import Required Libraries

Import the Hugging Face Hub library for downloading models from the Hugging Face repository.

In [3]:
from huggingface_hub import snapshot_download

## Step 4: Define Model Name

Specify the model we want to quantize. We're using the Qwen3.5-2B model from Hugging Face.

In [4]:
model_name = "Qwen/Qwen3.5-2B"

## Step 5: Define Quantization Methods

Specify the quantization method to use. 'q4_k_m' is a popular method that provides good balance between model size and quality.

In [5]:
methods = ['q4_k_m']

## Step 6: Define Directory Paths

Set up the directory paths where the original model will be stored and where quantized models will be saved.

In [6]:
base_model = "./original_model/"
quantized_path = "./quantized_model/"

## Step 7: Download Model from Hugging Face

Download the Qwen3.5-2B model from Hugging Face Hub to the local directory. This creates the original model files needed for conversion.

In [7]:
snapshot_download(repo_id=model_name, local_dir=base_model , local_dir_use_symlinks=False)
original_model = quantized_path+'/FP16.gguf'

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

## Step 8: Create Output Directory

Create the output directory where quantized models will be saved.

In [8]:
import os
os.makedirs('./quantized_model', exist_ok=True)

## Step 9: Convert Model to GGUF Format

Convert the model from Hugging Face format to GGUF format in FP16 (16-bit floating point) precision. This is the intermediate step before quantization.

In [9]:
!python llama.cpp/convert_hf_to_gguf.py ./original_model/ --outtype f16 --outfile ./quantized_model/FP16.gguf

INFO:hf-to-gguf:Loading model: original_model

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`
INFO:hf-to-gguf:Model architecture: Qwen3_5ForConditionalGeneration

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model pa

## Step 10: Build llama.cpp Project

Build the llama.cpp project using CMake. This compiles the C++ tools needed for quantization, including the llama-quantize executable.

In [ ]:
# Navigate to llama.cpp directory, create build directory, and use CMake to build
! cd llama.cpp && mkdir -p build && cd build && cmake .. && cmake --build . --config Release

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

## Step 11: Quantize the Model

Quantize the GGUF model from FP16 to the specified quantization method (q4_k_m). This significantly reduces the model size while maintaining reasonable performance.

In [11]:
import os
for m in methods:
    qtype = f"{quantized_path}/{m.upper()}.gguf"
    # Ensure the correct path to the llama-quantize executable built by CMake
    quantize_exec = "/content/llama.cpp/build/bin/llama-quantize"
    if not os.path.exists(quantize_exec):
        print(f"Error: llama-quantize executable not found at {quantize_exec}")
    else:
        command = f"{quantize_exec} {quantized_path}/FP16.gguf {qtype} {m}"
        print(f"Running quantization: {command}")
        os.system(command)


Running quantization: /content/llama.cpp/build/bin/llama-quantize ./quantized_model//FP16.gguf ./quantized_model//Q4_K_M.gguf q4_k_m


## Step 12: Set Up Prompt Files

Create a directory for prompt templates and create a sample prompt file for testing the model. This will be used to interact with the quantized model.

In [12]:
import os
prompt_dir = '/content/llama.cpp/prompts'
prompt_file = os.path.join(prompt_dir, 'chat-with-bob.txt')

if not os.path.exists(prompt_dir):
    print(f"Creating directory: {prompt_dir}")
    os.makedirs(prompt_dir)

if not os.path.exists(prompt_file):
    print(f"Creating placeholder prompt file: {prompt_file}")
    with open(prompt_file, 'w') as f:
        f.write("You are Bob, a helpful and friendly assistant. Say hello.\nUser: Hello, Bob.\nBob:")
else:
    print(f"Prompt file already exists: {prompt_file}")

# Also list the contents of the prompts directory to double check
print(f"\nContents of {prompt_dir}:")
!ls -l {prompt_dir}

Creating directory: /content/llama.cpp/prompts
Creating placeholder prompt file: /content/llama.cpp/prompts/chat-with-bob.txt

Contents of /content/llama.cpp/prompts:
total 4
-rw-r--r-- 1 root root 80 Apr 14 18:40 chat-with-bob.txt


## Step 13: Test the Quantized Model

Run the quantized model using the llama-cli tool. This tests that the quantized model works correctly by generating text based on the prompt file.

In [13]:
! /content/llama.cpp/build/bin/llama-cli -m /content/quantized_model/Q4_K_M.gguf -n 90 --repeat_penalty 1.0 --color auto -r "User:" -f /content/llama.cpp/prompts/chat-with-bob.txt


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/- 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8796-fae3a2807
model      : Q4_K_M.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern


> You are Bob, a helpful and friendly assistant. Say hello.
User: Hello, Bob.
Bob:

|-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/ [Start thinking]
Thinking Process:

1.  **Analyze the Requ

## Step 14: Authenticate with Hugging Face Hub

Log in to your Hugging Face account. This is required to upload models to the Hugging Face Hub repository.

In [14]:
from huggingface_hub import notebook_login
notebook_login()

## Step 15: Create Repository on Hugging Face Hub

Create a new repository on the Hugging Face Hub where the quantized model will be stored. This sets up the space where others can download the model.

In [17]:
from huggingface_hub import HfApi

model_path = "/content/quantized_model/Q4_K_M.gguf" # Your model's local path
repo_name = "qwen3.5-2b-llm-gguf"  # Desired HF Hub repository name

api = HfApi()
repo_url = api.create_repo(repo_name, private=False)

## Step 16: Upload Quantized Model to Hugging Face Hub

Upload the quantized Q4_K_M model to the Hugging Face Hub repository. This makes the model publicly available for others to download and use.

**Note:** Remember to replace the authentication token with your actual Hugging Face write token for security.

In [ ]:
from huggingface_hub import HfApi

model_path = "/content/quantized_model/Q4_K_M.gguf" # Your model's local path
write_token = "your_write_token_here"  # Replace "your_write_token_here" with your actual write token

api = HfApi()
api.upload_file(
    path_or_fileobj=model_path,
    path_in_repo="Q4_K_M.gguf",
    repo_id="SyedAffan/qwen3.5-2b-llm-gguf"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...antized_model/Q4_K_M.gguf:   8%|7         | 96.0MB / 1.27GB            

CommitInfo(commit_url='https://huggingface.co/SyedAffan/qwen3.5-2b-llm-gguf/commit/8f93fea012df66ac26924848eb458274fc2dfb41', commit_message='Upload Q4_K_M.gguf with huggingface_hub', commit_description='', oid='8f93fea012df66ac26924848eb458274fc2dfb41', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SyedAffan/qwen3.5-2b-llm-gguf', endpoint='https://huggingface.co', repo_type='model', repo_id='SyedAffan/qwen3.5-2b-llm-gguf'), pr_revision=None, pr_num=None)